In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50

torch.backends.cudnn.benchmark = True

DATASET_DIR = "/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset/train"

class Config:
    dataset_dir = DATASET_DIR
    output_dir = "/kaggle/working/simclr_checkpoints"
    batch_size = 64
    epochs = 100
    temperature = 0.5
    learning_rate = 1e-3
    embedding_dim = 128
    image_size = 224
    num_workers = 2
    seed = 42

config = Config()

os.makedirs(config.output_dir, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(config.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [3]:
class SimCLRTransform:
    def __init__(self, image_size=224):

        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(
                image_size,
                scale=(0.5, 1.0)
            ),

            transforms.RandomHorizontalFlip(),

            transforms.ColorJitter(
                brightness=0.4,
                contrast=0.4,
                saturation=0.4,
                hue=0.1
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            )
        ])

    def __call__(self, image):

        return (
            self.transform(image),
            self.transform(image)
        )

simclr_transform = SimCLRTransform(
    image_size=config.image_size
)

In [4]:
class SimCLRDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        valid_extensions = {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp",
            ".tif",
            ".tiff"
        }

        self.image_paths = sorted([
            path for path in self.root_dir.rglob("*")
            if path.suffix.lower() in valid_extensions
        ])

        self.identifiers = [
            path.stem
            for path in self.image_paths
        ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        view_i, view_j = self.transform(image)

        identifier = self.identifiers[idx]

        return view_i, view_j, identifier

train_dataset = SimCLRDataset(
    root_dir=config.dataset_dir,
    transform=simclr_transform
)

print(f"Total Images Found: {len(train_dataset)}")

if len(train_dataset) == 0:
    raise ValueError("No images found in dataset directory")

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
    drop_last=True
)

Total Images Found: 19186


In [5]:
class SimCLR(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()

        backbone = resnet50(weights=None)

        feature_dim = backbone.fc.in_features

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.projector = nn.Sequential(
            nn.Linear(feature_dim, feature_dim, bias=False),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feature_dim, embedding_dim, bias=False)
        )

    def forward(self, x, return_features=False):
        features = self.backbone(x)

        projections = self.projector(features)

        projections = F.normalize(
            projections,
            dim=1
        )

        if return_features:
            return features, projections

        return projections

model = SimCLR(
    embedding_dim=config.embedding_dim
)

In [6]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()

        self.temperature = temperature

    def forward(self, z_i, z_j):
        batch_size = z_i.size(0)

        representations = torch.cat(
            [z_i, z_j],
            dim=0
        )

        similarity_matrix = F.cosine_similarity(
            representations.unsqueeze(1),
            representations.unsqueeze(0),
            dim=2
        )

        mask = torch.eye(
            2 * batch_size,
            dtype=torch.bool,
            device=representations.device
        )

        similarity_matrix = similarity_matrix.masked_fill(
            mask,
            float("-inf")
        )

        positives = torch.cat([
            torch.diag(similarity_matrix, batch_size),
            torch.diag(similarity_matrix, -batch_size)
        ]).view(2 * batch_size, 1)

        negatives = similarity_matrix[
            ~mask
        ].view(2 * batch_size, -1)

        logits = torch.cat(
            [positives, negatives],
            dim=1
        )

        logits = logits / self.temperature

        labels = torch.zeros(
            2 * batch_size,
            dtype=torch.long,
            device=representations.device
        )

        loss = F.cross_entropy(
            logits,
            labels
        )

        return loss

criterion = NTXentLoss(
    temperature=config.temperature
)

In [7]:
model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=1e-6
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.epochs
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=device.type == "cuda"
)

In [8]:
from tqdm.auto import tqdm

for epoch in range(config.epochs):

    model.train()

    total_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{config.epochs}",
        leave=True
    )

    for batch_idx, (view_i, view_j, identifiers) in enumerate(progress_bar):

        view_i = view_i.to(
            device,
            non_blocking=True
        )

        view_j = view_j.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=device.type == "cuda"
        ):

            z_i = model(view_i)

            z_j = model(view_j)

            loss = criterion(
                z_i,
                z_j
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

        avg_loss = total_loss / (batch_idx + 1)

        progress_bar.set_postfix(
            avg_loss=f"{avg_loss:.4f}"
        )

    scheduler.step()

    epoch_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{config.epochs} "
        f"Average Loss: {epoch_loss:.4f}"
    )

    if (epoch + 1) % 20 == 0:

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "backbone_state_dict": model.backbone.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict()
        }

        torch.save(
            checkpoint,
            os.path.join(
                config.output_dir,
                f"simclr_epoch_{epoch + 1}.pt"
            )
        )

        torch.save(
            model.backbone.state_dict(),
            os.path.join(
                config.output_dir,
                f"resnet50_backbone_epoch_{epoch + 1}.pth"
            )
        )

        print(
            f"Checkpoint saved at epoch {epoch + 1}"
        )

Epoch 1/100:   0%|          | 0/299 [00:00<?, ?it/s]

Epoch 1/100 Average Loss: 3.9084


Epoch 2/100:   0%|          | 0/299 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d64ad17dc60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d64ad17dc60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch 2/100 Average Loss: 3.5116


...

Epoch 99/100:   0%|          | 0/299 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d64ad17dc60>
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d64ad17dc60>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        self._shutdown_workers()if w.is_alive():

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      if w.is_alive():  
    ^^ ^ ^  ^^^ ^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 Exception ignored in: ^<

Epoch 99/100 Average Loss: 2.9685


Epoch 100/100:   0%|          | 0/299 [00:00<?, ?it/s]

Epoch 100/100 Average Loss: 2.9680
Checkpoint saved at epoch 100
